# U2-04 · Árboles, SVM, redes neuronales y conjuntos de modelos

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 2, apartados 2.4 y 2.5*

Ejecuta cada celda con **Mayús + Intro**. En el navegador los modelos tardan más que en un ordenador con Python instalado: la SVM y el bosque aleatorio, alrededor de un minuto cada uno.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su origen se explica en `LEEME.md`:
> Janosi, A., Steinbrunn, W., Pfisterer, M. y Detrano, R. (1989). *Heart Disease*. https://doi.org/10.24432/C52P4X ·
> Hofmann, H. (1994). *Statlog (German Credit Data)*. https://doi.org/10.24432/C5NC77 ·
> Sakar, C. O. y Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset*. https://doi.org/10.24432/C5F88Q ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Caso de empresa: ¿comprará esta visita? (2.4)

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

d = pd.read_csv("compras.csv")                 # 12 330 visitas
y = d.pop("Revenue").astype(int)               # 1 = acabó en compra
cat = ["Month", "VisitorType", "Weekend", "OperatingSystems",
       "Browser", "Region", "TrafficType"]
num = d.columns.difference(cat)

def prep(escalar):              # los árboles no necesitan escalar
    return make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         cat),
        (StandardScaler() if escalar else "passthrough", num),
        verbose_feature_names_out=False)

X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

def evalua(nombre, mod):
    """Entrena y muestra el acierto equilibrado y el AUC en prueba."""
    t = time.perf_counter()
    mod.fit(X_ent, y_ent)
    if hasattr(mod, "decision_function"):            # la SVM
        s = mod.decision_function(X_pru)
    else:                                            # los demás
        s = mod.predict_proba(X_pru)[:, 1]
    eq = balanced_accuracy_score(y_pru, mod.predict(X_pru))
    print(f"{nombre:<22} entrenamiento {mod.score(X_ent, y_ent):.3f}"
          f" ({time.perf_counter() - t:.0f} s)")
    print(f"   prueba: acierto equilibrado {eq:.3f}, "
          f"AUC {roc_auc_score(y_pru, s):.3f}")
    return mod

if __name__ == "__main__":
    print(f"compran {y.mean():.1%} de las visitas")
    arbol = evalua("árbol, profundidad 3", make_pipeline(
        prep(False), DecisionTreeClassifier(max_depth=3,
                                            random_state=0)))
    evalua("árbol, sin límite", make_pipeline(
        prep(False), DecisionTreeClassifier(random_state=0)))
    evalua("SVM (núcleo RBF)", make_pipeline(prep(True), SVC()))
    evalua("red neuronal 32-16", make_pipeline(
        prep(True), MLPClassifier((32, 16), early_stopping=True,
                                  max_iter=500, random_state=0)))
    plt.figure(figsize=(5.5, 2.4))
    plot_tree(arbol[-1], max_depth=2, fontsize=5,
              feature_names=arbol[0].get_feature_names_out(),
              class_names=["no compra", "compra"], filled=True,
              impurity=False, proportion=True)
    plt.show()  # plt.savefig("arbol.pdf", bbox_inches="tight")

**Prueba.** Entrena árboles de profundidad 2, 4, 6 y 8. ¿En qué profundidad empieza a empeorar la prueba?

## 2. Bosque aleatorio, gradient boosting y votación (2.5)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

bosque = evalua("bosque aleatorio (500)", make_pipeline(
    prep(False), RandomForestClassifier(500, min_samples_leaf=3,
                                        n_jobs=-1, random_state=0)))
evalua("gradient boosting", make_pipeline(
    prep(False), HistGradientBoostingClassifier(random_state=0)))
votos = VotingClassifier([
    ("bosque", RandomForestClassifier(500, min_samples_leaf=3,
                                      n_jobs=-1, random_state=0)),
    ("logistica", LogisticRegression(max_iter=1000))], voting="soft")
evalua("votación (2 modelos)", make_pipeline(prep(True), votos))

# ¿qué variables usa el bosque? Se barajan una a una en prueba
imp = permutation_importance(bosque, X_pru, y_pru, scoring="roc_auc",
                             n_repeats=10, random_state=0)
imp = pd.Series(imp.importances_mean, index=X_pru.columns)
imp = imp.sort_values().tail(6)
print((imp * 100).round(1).to_string())       # puntos de AUC perdidos
imp.plot.barh(figsize=(4.4, 1.8))
plt.xlabel("pérdida de AUC al barajar la variable")
plt.show()  # plt.savefig("importancia.pdf")

## 3. Predicción avanzada de la serie de ventas (2.5)

La primera celda repite la preparación del apartado 2.3.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, r2_score,
                             root_mean_squared_error)

v = pd.read_csv("ventas_diarias.csv", parse_dates=["fecha"],
                index_col="fecha")["ventas"] / 1000  # miles de £

def variables(v):
    """Lo que se sabe la víspera: ventas pasadas y el calendario."""
    X = pd.DataFrame({"ayer": v.shift(1), "hace_7": v.shift(7),
                      "media_7": v.shift(1).rolling(7).mean(),
                      "media_28": v.shift(1).rolling(28).mean()})
    dias = pd.get_dummies(v.index.dayofweek, prefix="dia",
                          dtype=float).set_index(v.index)
    return X.join(dias).dropna()

X = variables(v)
y = v.loc[X.index]
ent, pru = X.index < "2011-07-01", X.index >= "2011-07-01"
print(f"entrenamiento {ent.sum()} días, prueba {pru.sum()} días")

def mide(nombre, pred):
    yp = y[pru]
    print(f"{nombre:<26} EAM {mean_absolute_error(yp, pred):5.2f}  "
          f"RECM {root_mean_squared_error(yp, pred):5.2f}  "
          f"R2 {r2_score(yp, pred):5.2f}")

if __name__ == "__main__":
    mide("referencia: hace 7 días", X.loc[pru, "hace_7"])
    lineal = LinearRegression().fit(X[ent], y[ent])
    mide("regresión lineal", lineal.predict(X[pru]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (RandomForestRegressor,
                              HistGradientBoostingRegressor)

X = X.assign(mes=X.index.month)                  # estación del año
modelos = {"regresión lineal": LinearRegression(),
           "bosque aleatorio": RandomForestRegressor(
               500, min_samples_leaf=3, random_state=0, n_jobs=-1),
           "gradient boosting": HistGradientBoostingRegressor(
               max_iter=300, learning_rate=0.05, random_state=0)}
mide("referencia: hace 7 días", X.loc[pru, "hace_7"])
pred = {}
for nombre, m in modelos.items():
    pred[nombre] = m.fit(X[ent], y[ent]).predict(X[pru])
    mide(nombre, pred[nombre])
media = (pred["bosque aleatorio"] + pred["gradient boosting"]) / 2
mide("media de los dos conjuntos", media)

fig, eje = plt.subplots(figsize=(4.4, 2.0))
eje.plot(y[pru].index, y[pru], lw=0.8, label="ventas reales")
eje.plot(y[pru].index, media, lw=0.8, label="previsión")
eje.set_ylabel("miles de £ al día"); eje.legend()
fig.autofmt_xdate(); plt.show()  # fig.savefig("ventas.pdf")

---
## Listados de ampliación

Los listados siguientes completan los apartados del libro, en el mismo orden. Los que reutilizan un caso anterior lo importan de los ficheros `.py` que están junto a este cuaderno (por ejemplo, `from h_ventas import X, y`).

### Los hiperparámetros de la SVM  ·  `i4_svm.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from e_corazon import d, y, preparacion

rejilla = {"svc__C": np.logspace(-2, 3, 6),
           "svc__gamma": np.logspace(-4, 1, 6)}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
busca = GridSearchCV(make_pipeline(preparacion(), SVC()), rejilla,
                     cv=cv, scoring="roc_auc").fit(d, y)
r = pd.DataFrame(busca.cv_results_)
tabla = r.pivot(index="param_svc__C", columns="param_svc__gamma",
                values="mean_test_score")
print("mejor:", {k[5:]: float(v) for k, v in
                 busca.best_params_.items()},
      f"AUC {busca.best_score_:.3f}")
print("AUC con gamma 10:", tabla.iloc[:, -1].round(3).tolist())
fig, eje = plt.subplots(figsize=(3.2, 2.4))
im = eje.imshow(tabla, cmap="viridis", origin="lower")
pot = lambda v: f"$10^{{{np.log10(v):.0f}}}$"   # 10 elevado a
eje.set_xticks(range(6), [pot(g) for g in tabla.columns])
eje.set_yticks(range(6), [pot(c) for c in tabla.index])
eje.set_xlabel("gamma"); eje.set_ylabel("C")
fig.colorbar(im, label="AUC de validación"); plt.show()  # fig.savefig("svm.pdf")

### El tamaño de una red neuronal  ·  `i5_red.py`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

for capas, alfa in [((8,), 1e-4), ((64,), 1e-4), ((64, 64), 1e-4),
                    ((64, 64), 1.0), ((256, 256), 1e-4)]:
    red = make_pipeline(prep(True), MLPClassifier(
        capas, alpha=alfa, max_iter=300, early_stopping=True,
        random_state=0)).fit(X_ent, y_ent)
    mlp = red[-1]
    pesos = sum(w.size + b.size for w, b in
                zip(mlp.coefs_, mlp.intercepts_))
    auc = roc_auc_score(y_pru, red.predict_proba(X_pru)[:, 1])
    print(f"capas {str(capas):<10} alfa {alfa:<6g} pesos {pesos:>6}"
          f"  épocas {mlp.n_iter_:>3}  AUC {auc:.3f}")

### Vecinos más próximos y Bayes ingenuo  ·  `i2_vecinos.py`

In [ ]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

for nombre, m in [("Bayes ingenuo", GaussianNB())] + [
        (f"{k} vecinos", KNeighborsClassifier(k)) for k in
        (1, 5, 25, 101)]:
    t = time.perf_counter()
    modelo = make_pipeline(prep(True), m).fit(X_ent, y_ent)
    p = modelo.predict_proba(X_pru)[:, 1]
    eq = balanced_accuracy_score(y_pru, modelo.predict(X_pru))
    seg = time.perf_counter() - t
    print(f"{nombre:<13} ent. {modelo.score(X_ent, y_ent):.3f}"
          f" | prueba: equil. {eq:.3f}, AUC "
          f"{roc_auc_score(y_pru, p):.3f} ({seg:.1f} s)")
# los vecinos sin escalar: manda la variable de mayor rango
sin = make_pipeline(prep(False), KNeighborsClassifier(25))
p = sin.fit(X_ent, y_ent).predict_proba(X_pru)[:, 1]
print(f"25 vecinos sin escalar: AUC {roc_auc_score(y_pru, p):.3f}")
print("rango de ProductRelated_Duration:",
      round(X_ent.ProductRelated_Duration.max()), "segundos")

### Podar un árbol  ·  `i3_poda.py`

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GridSearchCV
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

arbol = make_pipeline(prep(False),
                      DecisionTreeClassifier(random_state=0))
camino = arbol.fit(X_ent, y_ent)[-1].cost_complexity_pruning_path(
    arbol[0].transform(X_ent), y_ent)
print("hojas sin podar:", arbol[-1].get_n_leaves(),
      "| valores de alfa posibles:", len(camino.ccp_alphas))
rejilla = {"decisiontreeclassifier__ccp_alpha":
           np.quantile(camino.ccp_alphas, np.linspace(0.5, 1, 20))}
busca = GridSearchCV(arbol, rejilla, cv=5, scoring="roc_auc")
busca.fit(X_ent, y_ent)
podado = busca.best_estimator_
print(f"alfa elegido {podado[-1].ccp_alpha:.5f}: "
      f"{podado[-1].get_n_leaves()} hojas, "
      f"profundidad {podado[-1].get_depth()}")
for nombre, m in [("sin podar", arbol), ("podado", podado)]:
    print(f"{nombre:<10} acierto entrenamiento "
          f"{m.score(X_ent, y_ent):.3f}, prueba "
          f"{m.score(X_pru, y_pru):.3f}")
print(export_text(podado[-1], max_depth=1, decimals=2,
      feature_names=list(podado[0].get_feature_names_out())))

### Dos importancias que no coinciden  ·  `j7_mdi.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from e_corazon import d, y, X_ent, X_pru, y_ent, y_pru

X = d.fillna(d.median())                     # sin pipeline, a mano
rng = np.random.default_rng(0)
X = X.assign(azar=rng.normal(size=len(X)))   # columna sin relación
Xe, Xp = X.loc[X_ent.index], X.loc[X_pru.index]
rf = RandomForestClassifier(500, random_state=0).fit(Xe, y_ent)
mdi = pd.Series(rf.feature_importances_, X.columns)
perm = permutation_importance(rf, Xp, y_pru, n_repeats=30,
                              random_state=0, scoring="roc_auc")
perm = pd.Series(perm.importances_mean, X.columns)
tabla = pd.DataFrame({"impureza": mdi, "permutación": perm})
tabla["puesto impureza"] = mdi.rank(ascending=False).astype(int)
tabla["puesto perm."] = perm.rank(ascending=False).astype(int)
print(tabla.sort_values("impureza", ascending=False)
      .round(3).head(8).to_string())
print("puestos de 'azar':",
      tabla.loc["azar"].iloc[2:].astype(int).tolist())

### Ajustar los hiperparámetros  ·  `j2_busqueda.py`

In [ ]:
import time
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

modelo = make_pipeline(prep(False),
                       HistGradientBoostingClassifier(random_state=0))
espacio = {"histgradientboostingclassifier__" + k: v for k, v in {
    "learning_rate": loguniform(0.01, 0.3),
    "max_leaf_nodes": randint(4, 64),
    "min_samples_leaf": randint(10, 200),
    "l2_regularization": loguniform(1e-3, 10)}.items()}
t = time.perf_counter()
busca = RandomizedSearchCV(modelo, espacio, n_iter=30, cv=5,
                           scoring="roc_auc", random_state=0,
                           n_jobs=-1).fit(X_ent, y_ent)
print(f"30 combinaciones x 5 partes en "
      f"{time.perf_counter() - t:.0f} s")
r = pd.DataFrame(busca.cv_results_)
print("AUC de validación: peor", round(r.mean_test_score.min(), 3),
      "| mediana", round(r.mean_test_score.median(), 3),
      "| mejor", round(busca.best_score_, 3))
for k, v in busca.best_params_.items():
    print(" ", k.split("__")[1], round(v, 4))
base = modelo.fit(X_ent, y_ent).predict_proba(X_pru)[:, 1]
mejor = busca.predict_proba(X_pru)[:, 1]
print(f"AUC en prueba: por defecto {roc_auc_score(y_pru, base):.3f},"
      f" ajustado {roc_auc_score(y_pru, mejor):.3f}")

### Cómo responde el modelo a cada variable  ·  `j5_dependencia.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.inspection import partial_dependence
from i_compras import prep, X_ent, y_ent

gb = make_pipeline(prep(False),
                   HistGradientBoostingClassifier(random_state=0))
gb.fit(X_ent, y_ent)
for v, valores in [("PageValues", [0, 5, 20, 50]),
                   ("ExitRates", [0.005, 0.02, 0.05, 0.1])]:
    pd_ = partial_dependence(gb, X_ent, [v], custom_values={
        v: valores}, kind="average")
    print(v, dict(zip(valores, pd_["average"][0].round(3).tolist())))
fig, ejes = plt.subplots(1, 2, figsize=(4.4, 2.0), sharey=True)
PartialDependenceDisplay.from_estimator(
    gb, X_ent, ["PageValues", "ExitRates"], ax=ejes,
    kind="both", subsample=200, random_state=0,
    percentiles=(0.01, 0.95),
    ice_lines_kw={"alpha": 0.15, "lw": 0.4},
    pd_line_kw={"lw": 1.5, "c": "C1", "label": "media"})
ejes[0].set_ylabel("probabilidad de compra"); ejes[1].set_ylabel("")
plt.show()  # fig.savefig("dependencia.pdf")

### Gradient boosting árbol a árbol  ·  `j3_etapas.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

d = pd.read_csv("credito.csv")                 # 1000 clientes
y = (d.pop("riesgo") == 2).astype(int)         # 1 = mal pagador
cat = d.select_dtypes(exclude="number").columns
prep = make_column_transformer(
    (OneHotEncoder(drop="first"), cat),
    (StandardScaler(), d.columns.difference(cat)),
    sparse_threshold=0)
X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

Xe, Xp = prep.fit_transform(X_ent), prep.transform(X_pru)
fig, eje = plt.subplots(figsize=(4.4, 2.2))
for tasa, color in [(0.3, "C0"), (0.05, "C1")]:
    gb = GradientBoostingClassifier(n_estimators=600,
                                    learning_rate=tasa, max_depth=3,
                                    subsample=0.8, random_state=0)
    gb.fit(Xe, y_ent)
    ent = [log_loss(y_ent, p[:, 1])
           for p in gb.staged_predict_proba(Xe)]
    pru = [log_loss(y_pru, p[:, 1])
           for p in gb.staged_predict_proba(Xp)]
    mejor = int(np.argmin(pru)) + 1
    print(f"tasa {tasa}: mejor en prueba con {mejor} árboles "
          f"(pérdida {min(pru):.3f}); con 600: {pru[-1]:.3f}")
    eje.plot(pru, c=color, label=f"prueba, tasa {tasa}")
    eje.plot(ent, ":", c=color,
             label=f"entrenamiento, tasa {tasa}")
eje.set_xlabel("número de árboles"); eje.set_ylabel("log-loss")
eje.legend(fontsize=6); plt.show()  # fig.savefig("etapas.pdf")

### Apilar modelos  ·  `j4_apilar.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              StackingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

d = pd.read_csv("credito.csv")                 # 1000 clientes
y = (d.pop("riesgo") == 2).astype(int)         # 1 = mal pagador
cat = d.select_dtypes(exclude="number").columns
prep = make_column_transformer(
    (OneHotEncoder(drop="first"), cat),
    (StandardScaler(), d.columns.difference(cat)),
    sparse_threshold=0)

base = [("logistica", LogisticRegression(C=0.1, max_iter=1000)),
        ("bosque", RandomForestClassifier(300, min_samples_leaf=3,
                                          random_state=0)),
        ("vecinos", KNeighborsClassifier(25)),
        ("svm", SVC())]
apilado = StackingClassifier(base, LogisticRegression(), cv=5)
cv = StratifiedKFold(5, shuffle=True, random_state=1)
for nombre, m in base + [("apilado", apilado)]:
    auc = cross_val_score(make_pipeline(prep, m), d, y, cv=cv,
                          scoring="roc_auc")
    print(f"{nombre:<10} AUC {auc.mean():.3f} ± {auc.std():.3f}")
final = make_pipeline(prep, apilado).fit(d, y)[-1]
pesos = final.final_estimator_.coef_[0].round(2).tolist()
print("pesos del metamodelo:", dict(zip([n for n, _ in base], pesos)))

### Analizar los errores  ·  `k4_residuos.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from h_ventas import X, y, ent, pru

X = X.assign(mes=X.index.month)
m = RandomForestRegressor(500, min_samples_leaf=3, random_state=0,
                          n_jobs=-1).fit(X[ent], y[ent])
e = y[pru] - m.predict(X[pru])                 # real - previsto
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
por_dia = e.groupby(e.index.dayofweek).mean()
por_dia.index = [dias[i] for i in por_dia.index]
print("error medio por día (miles de £):")
print(por_dia.round(1).to_dict())
print("error medio por mes:",
      e.groupby(e.index.month).mean().round(1).to_dict())
grandes = e.abs().nlargest(3)
print("tres mayores errores:")
for f, v in grandes.items():
    print(f"  {f:%d/%m/%Y} real {y[f]:6.1f}  error {e[f]:+6.1f}")

### Por qué funcionan los conjuntos: el bagging  ·  `j6_bagging.py`

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from h_ventas import X, y, ent, pru

X = X.assign(mes=X.index.month)
def repetir(hacer, n=10):
    """Entrena n veces con muestras bootstrap distintas."""
    rng = np.random.default_rng(0)
    preds = []
    for _ in range(n):
        i = rng.integers(0, ent.sum(), ent.sum())
        m = hacer().fit(X[ent].iloc[i], y[ent].iloc[i])
        preds.append(m.predict(X[pru]))
    P = np.array(preds)
    eam = np.abs(P - y[pru].to_numpy()).mean(axis=1).mean()
    return eam, P.std(axis=0).mean()
for nombre, hacer in [
        ("un árbol", lambda: DecisionTreeRegressor(random_state=0)),
        ("bagging, 100 árboles", lambda: BaggingRegressor(
            DecisionTreeRegressor(), 100, random_state=0)),
        ("bosque, 100 árboles", lambda: RandomForestRegressor(
            100, max_features=0.33, random_state=0))]:
    eam, dispersion = repetir(hacer)
    print(f"{nombre:<21} EAM {eam:5.2f}  variación entre "
          f"repeticiones {dispersion:5.2f}")

### Prever un intervalo, no solo un número  ·  `k2_cuantiles.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from h_ventas import X, y, ent, pru

X = X.assign(mes=X.index.month)                 # como en 2.5
q = {}
for a in [0.1, 0.5, 0.9]:
    m = HistGradientBoostingRegressor(loss="quantile", quantile=a,
                                      max_iter=300,
                                      learning_rate=0.05,
                                      random_state=0)
    q[a] = m.fit(X[ent], y[ent]).predict(X[pru])
yp = y[pru].to_numpy()
dentro = (yp >= q[0.1]) & (yp <= q[0.9])
print(f"días dentro del intervalo 10-90 %: {dentro.mean():.0%}"
      f" (se esperaba 80 %)")
print(f"por debajo del 10 %: {(yp < q[0.1]).mean():.0%}, "
      f"por encima del 90 %: {(yp > q[0.9]).mean():.0%}")
print(f"anchura media del intervalo: "
      f"{(q[0.9] - q[0.1]).mean():.1f} miles de £")
fig, eje = plt.subplots(figsize=(4.4, 2.0))
eje.fill_between(y[pru].index, q[0.1], q[0.9], alpha=0.3,
                 lw=0, label="intervalo 10-90 %")
eje.plot(y[pru].index, q[0.5], lw=0.8, label="mediana prevista")
eje.plot(y[pru].index, yp, ".", ms=1.8, c="C1", label="real")
eje.set_ylabel("miles de £ al día"); eje.legend(fontsize=6)
fig.autofmt_xdate(); plt.show()  # fig.savefig("cuantiles.pdf")

### ¿Reentrenar cada mes?  ·  `k5_reentrenar.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from h_ventas import X, y, pru

X = X.assign(mes=X.index.month)
def bosque():
    return RandomForestRegressor(300, min_samples_leaf=3,
                                 random_state=0, n_jobs=-1)
fijo = bosque().fit(X[~pru], y[~pru]).predict(X[pru])
reent = []
for mes, grupo in X[pru].groupby(X[pru].index.to_period("M")):
    hasta = X.index < grupo.index[0]         # todo lo anterior
    m = bosque().fit(X[hasta], y[hasta])
    reent.append(pd.Series(m.predict(grupo), grupo.index))
reent = pd.concat(reent)
real = y[pru]
for nombre, p in [("entrenado una vez", fijo),
                  ("reentrenado cada mes", reent)]:
    print(f"{nombre:<21} EAM {mean_absolute_error(real, p):.2f}")
err = pd.DataFrame({"fijo": (real - fijo).abs(),
                    "mensual": (real - reent).abs()})
tabla = err.groupby(err.index.month).mean().round(1).T
tabla.columns.name = "mes"
print(tabla.to_string())

**Ejercicio.** Cambia el umbral de decisión del bosque aleatorio de la sección 2 para que su acierto equilibrado sea máximo. ¿Supera al árbol de profundidad 3?

> Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324